# Slicing a 3-D region of attraction (Kuramoto, n = 4)

With **four** Kuramoto oscillators the reduced state is `d = 3`, so the certified
region of attraction is a 3-D union of cubes — it cannot be plotted directly.
This notebook certifies that region and then visualizes it with
`pyddrv.viz.plot_roa_slice`: the **exact cross-section** of the union with an
axis-aligned 2-D plane. Requires the `[jax]` extra.

> Prerequisite (Python 3.10+): `pip install "pyddrv[jax,examples] @ git+https://github.com/NetDLab/pyDDRV"`
> (the `examples` extra bundles matplotlib + JupyterLab). See `examples/notebooks/README.md`.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from pyddrv import verify_roa
from pyddrv.systems.fields_jax import kuramoto_reduced_jax
from pyddrv.viz import plot_roa_slice

## 1. Certify the 3-D region

Same protocol as the `n = 3` notebook (closed-form `L`, max-norm, target rate
`alpha = 1`), one dimension up. We run **pass 1 only** (`trim=False`) with a
90-second budget — fast and right for exploration; for a *claimed* result,
re-run with `trim=True` (the two-pass protocol; in 3-D the trim raster is where
the cost is).

In [ ]:
k, n = 10.0, 4
f, L_bound = kuramoto_reduced_jax(k=k, n=n)
R, d = np.pi, n - 1                       # d = 3

t0 = time.time()
roa = verify_roa(f, R=R, d=d, alpha=1.0, L=L_bound, norm="inf",
                 tau=1.9, eps=np.pi/27, max_refine=4, max_seconds=90)
print(roa.summary())
print(f"certified fraction of Q_R: {roa.volume/(2*R)**d:.1%}, "
      f"wall {time.time()-t0:.0f}s")

Expect roughly **65% of `Q_pi`** certified in well under two minutes — about a
million cubes. `roa.centers` is now `(N, 3)`.

## 2. Slice it

`plot_roa_slice(roa, dims, at)` draws the cubes the plane
`{x_j = at[j] for j not in dims}` actually passes through (`|c_j - at_j| <= h`
on every fixed axis), projected onto the `dims` plane. It is an exact
cross-section — not a projection or a shadow. Default `at` is the equilibrium.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
for ax, phi3 in zip(axes, [0.0, 1.2, 2.4]):
    plot_roa_slice(roa, dims=(0, 1), at=[0.0, 0.0, phi3], ax=ax)
    ax.set_xlabel(r"$\phi_1$"); ax.set_ylabel(r"$\phi_2$")
    ax.set_title(rf"slice at $\phi_3 = {phi3:g}$")
fig.suptitle(f"Kuramoto n={n} (d=3): slices of the certified 1-RoA", y=1.02)
fig.tight_layout()
plt.show()

Reading the panels:

- **`phi_3 = 0`** (through the equilibrium): the familiar two-lobed sync-basin
  cross-section, with the excluded ball `B_eps` visible as the hole at the
  origin.
- **`phi_3 = 1.2`**: the section deforms and the hole disappears (the plane no
  longer meets `B_eps`); oscillator 3 being phase-advanced shifts which
  configurations are certifiably attracted.
- **`phi_3 = 2.4`**: near the edge of the certified set only a small
  off-center patch survives.

In every panel the layered grid is legible: coarse cubes deep inside, a dark
rim of fine cubes resolving the curved boundary.

## Takeaways

- `verify_roa` scales past `d = 2` unchanged; only the *plotting* needs the
  slice helper.
- A slice's 2-D area is **not** the region's volume — report `roa.volume`.
- Any plane works: `dims=(0, 2)` slices along `phi_1`–`phi_3`; `at` places it.